# Stage 05 - Compare Gradient Boosting

Train an optional LightGBM candidate against the same split and governed feature contract.

In [ ]:
from contextlib import nullcontext
from pathlib import Path
import importlib.util
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score


from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
FEATURE_TABLE = "silver_readiness_feature_store"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(FEATURE_TABLE):
    frame = spark_session.table(FEATURE_TABLE).toPandas()
    data_path = f"Lakehouse table {FEATURE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
encoded = pd.get_dummies(frame[feature_columns], columns=["maintenance_age_category", "test_phase"], dtype=float)
run_order = frame.groupby("simulation_run_id")["feature_timestamp_utc"].min().sort_values().index.tolist()
train_mask = frame["simulation_run_id"].isin(run_order[:2])
validation_mask = frame["simulation_run_id"].isin(run_order[2:3])
test_mask = frame["simulation_run_id"].isin(run_order[3:])

X_train = encoded.loc[train_mask].reset_index(drop=True)
y_train = frame.loc[train_mask, "synthetic_review_priority_label"].reset_index(drop=True)
X_validation = encoded.loc[validation_mask].reset_index(drop=True)
y_validation = frame.loc[validation_mask, "synthetic_review_priority_label"].reset_index(drop=True)
X_test = encoded.loc[test_mask].reset_index(drop=True)
y_test = frame.loc[test_mask, "synthetic_review_priority_label"].reset_index(drop=True)

LIGHTGBM_AVAILABLE = importlib.util.find_spec("lightgbm") is not None
MLFLOW_AVAILABLE = importlib.util.find_spec("mlflow") is not None
print(json.dumps({
    "data_path": str(data_path),
    "lightgbm_available": LIGHTGBM_AVAILABLE,
    "mlflow_available": MLFLOW_AVAILABLE,
}, indent=2))


In [ ]:
def prevalence_rank(probabilities, positive_rate):
    top_n = max(1, int(round(len(probabilities) * positive_rate)))
    ranking = pd.Series(probabilities).rank(method="first", ascending=False)
    return (ranking <= top_n).astype(int)


def metric_row(model_name, split_name, y_true, probabilities, positive_rate):
    y_true = pd.Series(y_true).astype(int)
    predicted = prevalence_rank(probabilities, positive_rate)
    return {
        "model_name": model_name,
        "split": split_name,
        "roc_auc": round(float(roc_auc_score(y_true, probabilities)), 4),
        "average_precision": round(float(average_precision_score(y_true, probabilities)), 4),
        "brier_loss": round(float(brier_score_loss(y_true, probabilities)), 4),
        "predicted_priority_rate": round(float(predicted.mean()), 4),
    }


if LIGHTGBM_AVAILABLE:
    from lightgbm import LGBMClassifier

    model = LGBMClassifier(
        n_estimators=120,
        learning_rate=0.05,
        num_leaves=15,
        random_state=7,
    )
    implementation = "lightgbm"
    implementation_note = "LightGBM package detected. Trained the optional advanced model."
else:
    model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=160, random_state=7)
    implementation = "hist-gradient-boosting-fallback"
    implementation_note = "LightGBM package not detected. Used a documented scikit-learn fallback instead."

mlflow = None
if MLFLOW_AVAILABLE:
    import mlflow

train_positive_rate = float(y_train.mean())
run_context = mlflow.start_run(run_name=f"demo04_{implementation}", nested=True) if mlflow is not None else nullcontext()
with run_context as active_run:
    model.fit(X_train, y_train)
    validation_probabilities = model.predict_proba(X_validation)[:, 1]
    test_probabilities = model.predict_proba(X_test)[:, 1]
    advanced_results = pd.DataFrame(
        [
            metric_row(implementation, "validation", y_validation, validation_probabilities, train_positive_rate),
            metric_row(implementation, "test", y_test, test_probabilities, train_positive_rate),
        ]
    )
    mlflow_run_id = None
    if active_run is not None:
        mlflow_run_id = active_run.info.run_id
        mlflow.log_params({
            "implementation": implementation,
            "train_runs": ",".join(run_order[:2]),
            "validation_run": run_order[2],
            "test_run": run_order[3],
            "feature_count": X_train.shape[1],
        })
        for row in advanced_results.to_dict(orient="records"):
            mlflow.log_metric(f"{row['split']}_roc_auc", row["roc_auc"])
            mlflow.log_metric(f"{row['split']}_average_precision", row["average_precision"])
            mlflow.log_metric(f"{row['split']}_brier_loss", row["brier_loss"])

advanced_results


In [ ]:
importance = permutation_importance(model, X_validation, y_validation, n_repeats=20, random_state=7)
importance_frame = pd.DataFrame(
    {
        "feature": X_validation.columns,
        "importance_mean": importance.importances_mean,
    }
).sort_values("importance_mean", ascending=False).reset_index(drop=True)

advanced_model_summary = {
    "classification": "SYNTHETIC_UNCLASS",
    "implementation": implementation,
    "implementation_note": implementation_note,
    "mlflow_run_id": mlflow_run_id,
    "lineage": {
        "feature_snapshot_ids": sorted(frame["feature_snapshot_id"].unique().tolist()),
        "source_snapshot_ids": sorted(frame["source_snapshot_id"].unique().tolist()),
    },
    "limitations": [
        "The advanced model remains optional because LightGBM is not guaranteed in every runtime.",
        "The fallback path is a teaching convenience, not a claim of equivalent behavior.",
        "Outputs remain analyst-support signals only.",
    ],
}

print(json.dumps(advanced_model_summary, indent=2))
importance_frame.head(10)
